# PostgreSQL mirror: joins, aggregations and window functions

Bonus criterion: SQL with joins and aggregations from Python on PostgreSQL, **in addition to** SQLite. SQLite (`data/processed/aircraft.sqlite`) stays the primary project database; PostgreSQL holds a mirror built from it by `src/postgres_mirror.py` with explicit types, primary keys and foreign keys. No result here may differ from SQLite where both can compute it; section 1 checks this.

Before running: start PostgreSQL, set `DATABASE_URL` (environment or the project's `.env` file) and run `python src/postgres_mirror.py` after every `src/prepare.py`. See `docs/POSTGRESQL.md` (macOS and Windows). The URL is never printed with its password.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.postgres_mirror import TABLES, engine, load_database_url, redact
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
URL = load_database_url()  # environment first, then the project's .env file
pg = engine(URL)
try:
    with pg.connect() as connection:
        server = connection.execute(text("SELECT version()")).scalar()
        mirrored = set(connection.execute(text(
            "SELECT tablename FROM pg_tables WHERE schemaname = current_schema()")).scalars())
except Exception as error:
    raise RuntimeError(f"Cannot reach PostgreSQL at {redact(URL)}. Start the server (docs/POSTGRESQL.md).") from None
CORE = ["airports", "observations", "aircraft_metadata", "observation_wind"]
if set(CORE) - mirrored:
    raise RuntimeError(f"Tables {sorted(set(CORE) - mirrored)} are missing in PostgreSQL. "
                       "Rebuild with src/prepare.py, then run python src/postgres_mirror.py.")
print("Connected to", redact(URL)); print(server)
ZONE_ORDER = "CASE zone WHEN 'near' THEN 1 WHEN 'middle' THEN 2 ELSE 3 END"
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


def pg_query(sql, **params):
    """Run SQL in PostgreSQL from Python; parameters are bound, never pasted into the string."""
    return pd.read_sql_query(text(sql), pg, params=params)


## 1. Same data as SQLite

Row counts of every mirrored table, then one identical SQL aggregation run in both databases. If anything differs, the mirror is out of date: rerun `python src/postgres_mirror.py`.

In [ ]:
counts = []
with closing(sqlite3.connect(DB)) as db, pg.connect() as connection:
    sqlite_tables = {name for (name,) in db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    for table in TABLES:
        counts.append({"table": table,
            "sqlite_rows": db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0] if table in sqlite_tables else None,
            "postgres_rows": connection.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar() if table in mirrored else None})
counts = pd.DataFrame(counts)
counts["match"] = counts.sqlite_rows.eq(counts.postgres_rows)
display(counts)
counts.to_csv(OUT / "tables/postgres_row_counts.csv", index=False)
assert counts.match.all(), "Mirror is out of date: run python src/postgres_mirror.py again."

# The same statement is valid in both dialects, so the results must agree.
same_sql = f"""SELECT zone, COUNT(*) AS observations, COUNT(DISTINCT icao24) AS aircraft,
                      AVG(velocity) AS mean_ground_speed_ms
               FROM observations GROUP BY zone ORDER BY {ZONE_ORDER}"""
with closing(sqlite3.connect(DB)) as db:
    zone_sqlite = pd.read_sql_query(same_sql, db)
zone_postgres = pg_query(same_sql)
pd.testing.assert_frame_equal(zone_sqlite, zone_postgres, check_dtype=False, rtol=1e-9)
display(zone_sqlite.merge(zone_postgres, on="zone", suffixes=("_sqlite", "_postgres")))


## 2. Join: observations – airports – wind

Three tables: every observation, its nearest selected airport (inner join, the airport key is enforced by a foreign key) and its modelled wind (left join, so observations without wind stay). `FILTER` restricts the wind averages to matched rows. The nearest selected airport is not an origin or destination.

In [ ]:
airport_wind = pg_query(f"""
    SELECT a.ident, a.name, o.zone,
           COUNT(*) AS observations,
           COUNT(DISTINCT o.icao24) AS aircraft,
           AVG(o.velocity) AS mean_ground_speed_ms,
           COUNT(*) FILTER (WHERE w.wind_matched = 1) AS wind_matched_observations,
           AVG(w.wind_speed_ms) FILTER (WHERE w.wind_matched = 1) AS mean_wind_speed_ms,
           AVG(w.tailwind_ms) FILTER (WHERE w.wind_matched = 1) AS mean_tailwind_ms
    FROM observations o
    JOIN airports a ON a.ident = o.airport_ident
    LEFT JOIN observation_wind w ON w.icao24 = o.icao24 AND w.time_position = o.time_position
    GROUP BY a.ident, a.name, o.zone
    ORDER BY a.ident, {ZONE_ORDER.replace('zone', 'o.zone')}""")
# The left join must not add or drop observations.
assert airport_wind.observations.sum() == counts.set_index("table").loc["observations", "postgres_rows"]
display(airport_wind.round(2))
airport_wind.to_csv(OUT / "tables/postgres_airport_zone_wind.csv", index=False)


### Interpretation (group): join

*Placeholder.* Which airport/zone groups have enough aircraft to compare? Is the mean tailwind similar across airports, or does one airport see mostly eastbound or westbound traffic? Means over observations weight frequently seen aircraft more.

## 3. Aggregation by zone and altitude band

Same 2000 m band edges as notebook 01 (-1000, 1000, 3000, … m, left-closed). PostgreSQL computes exact medians with `percentile_cont`, which SQLite does not offer.

In [ ]:
band_zone = pg_query(f"""
    SELECT -1000 + 2000 * FLOOR((o.geo_altitude + 1000) / 2000)::int AS band_low_m,
           o.zone,
           COUNT(*) AS observations,
           COUNT(DISTINCT o.icao24) AS aircraft,
           percentile_cont(0.5) WITHIN GROUP (ORDER BY o.velocity) AS median_ground_speed_ms,
           percentile_cont(0.5) WITHIN GROUP (ORDER BY w.tailwind_ms)
               FILTER (WHERE w.wind_matched = 1) AS median_tailwind_ms
    FROM observations o
    LEFT JOIN observation_wind w ON w.icao24 = o.icao24 AND w.time_position = o.time_position
    GROUP BY 1, 2
    ORDER BY 1, {ZONE_ORDER.replace('zone', 'o.zone')}""")
display(band_zone.round(1))
band_zone.to_csv(OUT / "tables/postgres_zone_altitude_band.csv", index=False)
MIN_AIRCRAFT = 5  # bands with fewer distinct aircraft are not drawn
plot_data = band_zone[band_zone.aircraft >= MIN_AIRCRAFT]
bands = sorted(band_zone.band_low_m.unique())
fig, ax = plt.subplots(figsize=(10, 5))
for zone, color in {"near": "#d97706", "middle": "#64748b", "far": "#2563eb"}.items():
    # Reindex over all bands: a band without enough aircraft leaves a gap instead of a misleading straight line.
    medians = plot_data[plot_data.zone == zone].set_index("band_low_m").median_ground_speed_ms.reindex(bands)
    ax.plot([band + 1000 for band in bands], medians.to_numpy(), marker="o", color=color, label=zone)
ax.set_xlabel("Geometric altitude, band centre (m)"); ax.set_ylabel("Median ground speed (m/s)")
ax.set_title(f"Median ground speed by altitude band and zone (PostgreSQL; bands with ≥ {MIN_AIRCRAFT} aircraft)")
ax.legend(title="Zone")
fig.tight_layout(); fig.savefig(OUT / "figures/postgres_median_speed_by_band.png", dpi=160); plt.show()


### Interpretation (group): zone and altitude band

*Placeholder.* Within the same altitude band, do near and far differ in median ground speed? How much of the overall zone difference is altitude mix? Compare with the median tailwind per band before attributing a difference to aircraft behaviour.

## 4. Window functions: fastest aircraft per airport

`ROW_NUMBER()` keeps each aircraft's fastest observation per nearest airport (so frequently observed aircraft count once), then `RANK()` orders the aircraft within each airport. Tailwind and altitude of that observation are shown because a high ground speed is often wind, not aircraft performance.

In [ ]:
TOP_N = 5
fastest = pg_query("""
    WITH per_observation AS (
        SELECT o.airport_ident, o.icao24, o.velocity, o.geo_altitude, w.tailwind_ms,
               ROW_NUMBER() OVER (PARTITION BY o.airport_ident, o.icao24
                                  ORDER BY o.velocity DESC, o.time_position) AS nth_fastest,
               COUNT(*) OVER (PARTITION BY o.airport_ident, o.icao24) AS observations
        FROM observations o
        LEFT JOIN observation_wind w ON w.icao24 = o.icao24 AND w.time_position = o.time_position),
    ranked AS (
        SELECT p.*, RANK() OVER (PARTITION BY airport_ident ORDER BY velocity DESC) AS speed_rank,
               COUNT(*) OVER (PARTITION BY airport_ident) AS aircraft_at_airport
        FROM per_observation p WHERE nth_fastest = 1)
    SELECT r.airport_ident, a.name, r.speed_rank, r.icao24,
           COALESCE(NULLIF(m.aircraft_typecode, ''), 'unknown/other') AS typecode,
           r.velocity AS max_ground_speed_ms, r.velocity * 3.6 AS max_ground_speed_kmh,
           r.geo_altitude, r.tailwind_ms, r.observations, r.aircraft_at_airport
    FROM ranked r
    JOIN airports a ON a.ident = r.airport_ident
    LEFT JOIN aircraft_metadata m ON m.icao24 = r.icao24
    WHERE r.speed_rank <= :top_n
    ORDER BY r.airport_ident, r.speed_rank""", top_n=TOP_N)
display(fastest.round(1))
fastest.to_csv(OUT / "tables/postgres_fastest_per_airport.csv", index=False)


### Interpretation (group): fastest aircraft

*Placeholder.* Are the fastest aircraft fast because of type or because of tailwind (compare `tailwind_ms`)? Ranks are descriptive and depend on which aircraft happened to be observed; ties share a rank. Do not name individual aircraft or operators in the presentation.